# Step 5 - Baseline Forecasting Models

Before testing advanced time-series models, we evaluate several **baseline methods**.
These baselines provide strong reference points and help quantify the real value added
by more complex approaches.

We implement three baselines:

1. **Naive (Persistence)**  
   $$
   \hat{y}_{t+h} = y_t
   $$
2. **Seasonal Naive (Weekly)**  
$$
   \hat{y}_{t+h} = y_{t+h-7}
$$
3. **7-day Moving Average**  
   Mean of the last 7 observed days

All baselines are evaluated using the **same expanding-window backtesting protocol**
defined in Step 4, for:
- **H = 1**
- **H = 7**


## 5.1 Baseline 1 : Naive (Persistence)

This baseline assumes that tomorrow’s demand will be identical to today’s demand.
It is often surprisingly competitive for short horizons.


In [19]:
def forecast_naive(train_df, horizon):
    last_obs = train_df.iloc[-1]

    future_index = pd.date_range(
        train_df.index.max() + pd.Timedelta(days=1),
        periods=horizon,
        freq="D"
    )

    pred = pd.DataFrame(
        np.tile(last_obs.values, (horizon, 1)),
        index=future_index,
        columns=train_df.columns
    )
    return pred


## 5.2 Baseline 2 : Seasonal Naive (Weekly)

This baseline exploits the strong weekly seasonality identified in Step 3.

Each forecasted day uses the observed demand from the same weekday one week earlier.
This is a very strong benchmark for retail and bakery demand.


In [20]:
def forecast_seasonal_naive(train_df, horizon, season_length=7):
    future_index = pd.date_range(
        train_df.index.max() + pd.Timedelta(days=1),
        periods=horizon,
        freq="D"
    )

    preds = []
    for h in range(1, horizon + 1):
        ref_date = train_df.index.max() + pd.Timedelta(days=h - season_length)
        if ref_date in train_df.index:
            preds.append(train_df.loc[ref_date].values)
        else:
            preds.append(train_df.iloc[-1].values)  # fallback

    pred = pd.DataFrame(preds, index=future_index, columns=train_df.columns)
    return pred


## 5.3 Baseline 3 : 7-day Moving Average

This baseline smooths short-term noise by averaging demand over the last 7 days.
It performs well when demand is stable but may lag during regime changes.

In [21]:
def forecast_ma7(train_df, horizon, window=7):
    hist = train_df.tail(window)
    avg = hist.mean(axis=0)

    future_index = pd.date_range(
        train_df.index.max() + pd.Timedelta(days=1),
        periods=horizon,
        freq="D"
    )

    pred = pd.DataFrame(
        np.tile(avg.values, (horizon, 1)),
        index=future_index,
        columns=train_df.columns
    )
    return pred


## 5.4 Run Baselines on Validation Set

We evaluate each baseline using the expanding-window protocol
defined in Step 4, for both horizons.


In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error


baselines = {
    "Naive": forecast_naive,
    "SeasonalNaive": forecast_seasonal_naive,
    "MA7": forecast_ma7
}

def time_split(df_ts, train_end, val_end):
    train_end = pd.to_datetime(train_end)
    val_end = pd.to_datetime(val_end)

    train = df_ts.loc[:train_end]
    val = df_ts.loc[train_end + pd.Timedelta(days=1):val_end]
    test = df_ts.loc[val_end + pd.Timedelta(days=1):]
    return train, val, test

TRAIN_END = "2022-03-31"
VAL_END   = "2022-06-30"

qty_ts = pd.read_csv("data_tmp/qty_ts.csv", index_col=0, parse_dates=True)

# 1) Sécuriser l'index
qty_ts = qty_ts.sort_index()
qty_ts = qty_ts[~qty_ts.index.duplicated(keep="first")]

# 2) Reconstituer tous les jours du min au max et remplir les jours manquants à 0
full_range = pd.date_range(qty_ts.index.min(), qty_ts.index.max(), freq="D")
qty_ts = qty_ts.reindex(full_range).fillna(0)
train_ts, val_ts, test_ts = time_split(qty_ts, TRAIN_END, VAL_END)
VAL_END_DATE = val_ts.index.max()

val_results = []

def backtest_expanding(full_df, forecast_fn, start_train_end, end_date, horizon=7, step=1):
    end_date = pd.to_datetime(end_date)

    rows = []
    for train_end, pred_start, pred_end in make_backtest_folds(full_df.index, start_train_end, horizon=horizon, step=step):
        if pred_end > end_date:
            break

        train_df = full_df.loc[:train_end]
        y_true = full_df.loc[pred_start:pred_end]
        y_pred = forecast_fn(train_df, horizon=horizon)

        # Defensive alignment
        y_pred = y_pred.reindex(index=y_true.index, columns=full_df.columns)

        for product in full_df.columns:
            yt = y_true[product].values
            yp = y_pred[product].values

            rows.append({
                "train_end": train_end,
                "pred_start": pred_start,
                "pred_end": pred_end,
                "horizon": horizon,
                "product": product,
                "MAE": mean_absolute_error(yt, yp),
                "RMSE": np.sqrt(mean_squared_error(yt, yp)),
            })

    return pd.DataFrame(rows)

def make_backtest_folds(index, start_train_end, horizon=7, step=1):
    start_train_end = pd.to_datetime(start_train_end)
    all_dates = pd.to_datetime(index)

    train_end = start_train_end
    last_date = all_dates.max()

    while True:
        pred_start = train_end + pd.Timedelta(days=1)
        pred_end = pred_start + pd.Timedelta(days=horizon - 1)
        if pred_end > last_date:
            break
        yield train_end, pred_start, pred_end
        train_end = train_end + pd.Timedelta(days=step)
 
for name, fn in baselines.items():
    for H in [1, 7]:
        scores = backtest_expanding(
            full_df=qty_ts,
            forecast_fn=fn,
            start_train_end=TRAIN_END,
            end_date=VAL_END_DATE,
            horizon=H,
            step=1
        )
        scores["model"] = name
        val_results.append(scores)

val_scores = pd.concat(val_results, ignore_index=True)


In [23]:
val_summary = (
    val_scores
    .groupby(["model", "horizon"])[["MAE", "RMSE"]]
    .mean()
    .reset_index()
    .sort_values(["horizon", "MAE"])
)

display(val_summary)


,model,horizon,MAE,RMSE
2,Naive,1,13.837912,13.837912
4,SeasonalNaive,1,13.847070,13.847070
0,MA7,1,13.949895,13.949895
5,SeasonalNaive,7,14.307423,17.889802
1,MA7,7,14.773689,18.147072
3,Naive,7,18.318207,22.322584


## 5.5 Run Baselines on Test Set

After selecting baselines on validation, we apply the same evaluation
to the test period to estimate real-world performance.


In [24]:
TEST_END_DATE = test_ts.index.max()

test_results = []

for name, fn in baselines.items():
    for H in [1, 7]:
        scores = backtest_expanding(
            full_df=qty_ts,
            forecast_fn=fn,
            start_train_end=VAL_END,
            end_date=TEST_END_DATE,
            horizon=H,
            step=1
        )
        scores["model"] = name
        test_results.append(scores)

test_scores = pd.concat(test_results, ignore_index=True)


In [25]:
test_summary = (
    test_scores
    .groupby(["model", "horizon"])[["MAE", "RMSE"]]
    .mean()
    .reset_index()
    .sort_values(["horizon", "MAE"])
)

display(test_summary)


,model,horizon,MAE,RMSE
0,MA7,1,14.426760,14.426760
4,SeasonalNaive,1,15.723732,15.723732
2,Naive,1,15.767210,15.767210
5,SeasonalNaive,7,16.270764,19.454822
1,MA7,7,17.324711,21.378804
3,Naive,7,19.911545,23.747072


The baseline evaluation confirms the strong weekly seasonality observed in the exploratory analysis. For next-day forecasts (H=1), simple persistence and moving-average methods already achieve low errors, indicating high short-term autocorrelation. For weekly forecasts (H=7), the seasonal naïve baseline clearly outperforms other simple methods, demonstrating that weekday effects dominate demand dynamics. Performance degradation from validation to test remains moderate, suggesting that seasonal structure is stable despite level shifts over time. These results establish a strong benchmark that any advanced forecasting model must surpass.